In [0]:
%reload_ext autoreload
%autoreload 2

In [0]:
from utils.logger import get_logger
from etl_config.gold_dim_config import DIM_CONFIG

logger = get_logger("gold_dim")

In [0]:
dbutils.widgets.text("dim_name", "")
dim_name = dbutils.widgets.get("dim_name")
#dim_name = 'dim_category'

if not dim_name or dim_name not in DIM_CONFIG:
    raise ValueError(f"Invalid or missing dim_name widget: '{dim_name}'")

cfg = DIM_CONFIG[dim_name]
column_names = [c.name for c in cfg.business_columns]
logger.info(f"Starting Gold processing for: {dim_name}")

In [0]:
source_df = spark.sql(cfg.source_query).select(column_names)
source_df.createOrReplaceTempView(f"stg_{dim_name}")
logger.info(f"'{dim_name}': source query returned {source_df.count()} rows")

In [0]:
full_table_name = cfg.target_table
source_count = source_df.count()

if not cfg.tracked_columns:
    
    # dim_date
    
    merge_condition = f"target.{cfg.natural_key} = source.{cfg.natural_key}"

    non_key_columns = [c for c in column_names if c != cfg.natural_key]
    update_set = ", ".join([f"target.{c} = source.{c}" for c in non_key_columns])
    insert_cols = ", ".join(column_names)
    insert_vals = ", ".join([f"source.{c}" for c in column_names])

    spark.sql(f"""
        merge into {full_table_name} as target
        using stg_{dim_name} as source
        on {merge_condition}
        when matched then
            update set {update_set}
        when not matched then
            insert ({insert_cols}) values ({insert_vals})
    """)

    merge_metrics = spark.sql(f"describe history {full_table_name} limit 1") \
        .select("operationMetrics").collect()[0]["operationMetrics"]
    inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
    updated = int(merge_metrics.get("numTargetRowsUpdated", 0))

    logger.info(f"'{dim_name}': MERGE complete")
    logger.info(f"'{dim_name}': source={source_count}, inserted={inserted}, updated={updated}")

else:
    
    # SCD2 dimensions
    
    # capture run timestamp before any writes, used to identify records expired in THIS run
    run_ts = spark.sql("select current_timestamp() as ts").collect()[0]["ts"]
    logger.info(f"'{dim_name}': run timestamp captured: {run_ts}")

    # build change detection condition, handles NULL comparisons explicitly
    change_condition = " or ".join([
        f"(target.{c} != source.{c} or "
        f"(target.{c} is null and source.{c} is not null) or "
        f"(target.{c} is not null and source.{c} is null))"
        for c in cfg.tracked_columns
    ])

    insert_cols = ", ".join(column_names + ["valid_from", "valid_to", "is_current"])
    insert_vals = ", ".join(
        [f"source.{c}" for c in column_names]
        + ["current_timestamp()", "null", "true"]
    )

    # expire changed records + insert new natural keys

    spark.sql(f"""
        merge into {full_table_name} as target
        using stg_{dim_name} as source
        on target.{cfg.natural_key} = source.{cfg.natural_key}
        and target.is_current = true
        when matched and ({change_condition}) then
            update set
                target.valid_to   = current_timestamp(),
                target.is_current = false
        when not matched then
            insert ({insert_cols})
            values ({insert_vals})
    """)

    merge_metrics_scd2 = spark.sql(f"describe history {full_table_name} limit 1") \
        .select("operationMetrics").collect()[0]["operationMetrics"]
    new_keys_inserted = int(merge_metrics_scd2.get("numTargetRowsInserted", 0))
    expired_count = int(merge_metrics_scd2.get("numTargetRowsUpdated", 0))

    logger.info(f"'{dim_name}': Merge complete - expired changed records, inserted new keys")

    # insert new versions for records expired in THIS run
    # uses run_ts to identify only records expired in this run,
    # not records expired in previous runs

    spark.sql(f"""
        insert into {full_table_name} ({insert_cols})
        select
            {", ".join([f"source.{c}" for c in column_names])},
            current_timestamp() as valid_from,
            null                as valid_to,
            true                as is_current
        from stg_{dim_name} as source
        inner join {full_table_name} as target
            on source.{cfg.natural_key} = target.{cfg.natural_key}
            and target.is_current = false
            and target.valid_to >= '{run_ts}'
    """)

    insert_metrics_scd2 = spark.sql(f"describe history {full_table_name} limit 1") \
        .select("operationMetrics").collect()[0]["operationMetrics"]
    new_versions_inserted = int(insert_metrics_scd2.get("numTargetRowsInserted", 0))

    logger.info(f"'{dim_name}': Inserted new versions for changed records")
    logger.info(
        f"'{dim_name}': source={source_count}, new_keys={new_keys_inserted}, "
        f"expired={expired_count}, new_versions={new_versions_inserted}"
    )

logger.info(f"'{dim_name}': Processing complete into {full_table_name}")

### Summary

In [0]:
total_count = spark.table(full_table_name).count()

if cfg.tracked_columns:
    current_count = spark.table(full_table_name).filter("is_current = true").count()
    logger.info(f"=== Gold Dimension Summary: {dim_name} ===")
    logger.info(f"  Source rows (from Silver):   {source_count}")
    logger.info(f"  New natural keys inserted:   {new_keys_inserted}")
    logger.info(f"  Existing rows expired:       {expired_count}")
    logger.info(f"  New versions inserted:       {new_versions_inserted}")
    logger.info(f"  Total rows in table:         {total_count}")
    logger.info(f"  Current rows (is_current):   {current_count}")
    if expired_count != new_versions_inserted:
        logger.warning(
            f"  ! Mismatch: expired ({expired_count}) != new_versions ({new_versions_inserted})"
        )
    logger.info("=" * 40)
else:
    logger.info(f"=== Gold Dimension Summary: {dim_name} ===")
    logger.info(f"  Source rows (from Silver):   {source_count}")
    logger.info(f"  Inserted:                    {inserted}")
    logger.info(f"  Updated:                     {updated}")
    logger.info(f"  Total rows in table:         {total_count}")
    logger.info("============================================")
